# LinkedIn Profile Saver (Automatic Login)

This notebook **auto-logs into LinkedIn** using `EMAIL` and `PASSWORD` environment variables and **saves one profile's full HTML source** to the `saved_profiles/` folder.

**Chosen options:**
- Save format: **Full `.html` (page_source only)**
- Workflow: **One profile at a time** (enter URL, run cell to save)
- Browser mode: **Visible** (you will see the Chrome window)
- Retry logic: **No automatic retries**

**Security note:** Do **not** hard-code credentials in the notebook. Set `EMAIL` and `PASSWORD` as environment variables in your OS before running. Use this only with accounts you own and in compliance with LinkedIn's Terms of Service.

Run the cells in order.


In [1]:
import os
import re
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

SAVE_DIR = 'saved_profiles'
os.makedirs(SAVE_DIR, exist_ok=True)

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', '_', name)

def create_driver(visible=True):
    options = webdriver.ChromeOptions()
    if not visible:
        options.add_argument('--headless=new')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--start-maximized')
    # Use webdriver-manager to get a compatible chromedriver
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver


In [2]:
def linkedin_login(driver, email, password, pause_after=3):
    """
    Logs into LinkedIn using the provided Selenium driver and credentials.
    Returns True if login appears successful (checks for presence of profile icon or redirect), False otherwise.
    """
    login_url = 'https://www.linkedin.com/login'
    driver.get(login_url)
    time.sleep(2)
    try:
        email_el = driver.find_element(By.ID, 'username')
        pass_el = driver.find_element(By.ID, 'password')
    except Exception as e:
        print('Could not find login inputs:', e)
        return False

    email_el.clear(); email_el.send_keys(email)
    pass_el.clear(); pass_el.send_keys(password)
    pass_el.submit()
    time.sleep(pause_after)

    # Basic check: if URL contains '/feed' or there's no '/login' in current URL, assume success
    current = driver.current_url
    if '/login' in current:
        print('Still on login page; login may have failed or additional verification required.')
        return False
    print('Login appears to have succeeded; current URL:', current)
    return True


In [3]:
def save_profile_as_html(driver, profile_url, index=1, wait=5):
    driver.get(profile_url)
    time.sleep(wait)  # give the page some time to load dynamic content
    html_source = driver.page_source
    # Use the page title if available; fallback to a numbered filename
    try:
        title = sanitize_filename(driver.title.strip()) or f'profile_{index}'
    except Exception:
        title = f'profile_{index}'
    filename = os.path.join(SAVE_DIR, f"{title}.html")
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_source)
    print(f"✅ Saved HTML: {filename}")
    return filename


## Usage
1. Make sure `EMAIL` and `PASSWORD` environment variables are set in your OS/session.
   - Linux / macOS example:
     ```bash
     export EMAIL='your_email@example.com'
     export PASSWORD='your_password'
     ```
   - Windows PowerShell example:
     ```powershell
     $env:EMAIL = 'your_email@example.com'
     $env:PASSWORD = 'your_password'
     ```
2. Run the next cell to create a visible Chrome driver and log in.
3. Paste a single LinkedIn profile URL when prompted and run the final cell to save it.


In [4]:
# 2) Create driver and login (visible mode as requested)
EMAIL = os.getenv('EMAIL')
PASSWORD = os.getenv('PASSWORD')
if not EMAIL or not PASSWORD:
    raise EnvironmentError('EMAIL and PASSWORD environment variables must be set before running this cell.')

driver = create_driver(visible=True)
logged = linkedin_login(driver, EMAIL, PASSWORD)
if not logged:
    print('\nLogin did not complete successfully. Check for two-factor/auth challenge in the browser window. You may need to login manually and then re-run the save cell.')


Login appears to have succeeded; current URL: https://www.linkedin.com/feed/


In [5]:
import json, os

# ✅ Read URL from Flask-written JSON
url_file = "url_input.json"
if not os.path.exists(url_file):
    raise FileNotFoundError("url_input.json not found. Flask app must provide the URL first.")

with open(url_file, "r", encoding="utf-8") as f:
    data = json.load(f)
    url = data.get("url")

if not url:
    raise ValueError("No URL found in url_input.json")

print(f"🔗 Loaded URL from Flask: {url}")
driver.get(url)


🔗 Loaded URL from Flask: https://www.linkedin.com/in/nisarga-8221652a4/


In [6]:
import time

driver.get(url)
time.sleep(5)  # waits 5 seconds before grabbing the HTML

page_source = driver.page_source
filename = sanitize_filename(url.split("/")[-2] or "profile") + ".html"

with open(os.path.join(SAVE_DIR, filename), "w", encoding="utf-8") as f:
    f.write(page_source)

print(f"✅ Saved profile HTML to {SAVE_DIR}/{filename}")


✅ Saved profile HTML to saved_profiles/nisarga-8221652a4.html


In [7]:
import subprocess

# ✅ Automatically trigger parser script
try:
    print(" Running LinkedIn parser on saved profiles...")
    subprocess.run(["python", "parse_linkedin_02.py", SAVE_DIR], check=True)
    print(" Parsing complete. Output generated successfully.")
except subprocess.CalledProcessError as e:
    print(f" Error while running parse_linkedin_02.py: {e}")


 Running LinkedIn parser on saved profiles...


 Parsing complete. Output generated successfully.


### Troubleshooting tips
- If LinkedIn challenges the login (captcha/2FA), complete it manually in the visible browser window opened by the driver, then re-run the save cell.
- If `webdriver_manager` fails to install a driver, install ChromeDriver manually and adjust `Service()` in `create_driver()` to point to the binary.
- Keep requests slow and minimal to avoid LinkedIn's anti-bot measures.
